# Pipeline

https://github.com/datamindedbe/blog-tpcds-dbt-duckdb/tree/main

```
uv sync
```

In [ ]:
# # run this to generate index for values in the hierarchy yaml files

# import duckdb
# from src.hierarchy_duckdb import build_tree_with_stats
# from pathlib import Path
# proj_path = Path().resolve()
# data_path = proj_path / 'data'
# duckdb_conn = duckdb.connect(database=str(proj_path / 'tpcds/tpcds.db'))
# index_path = data_path / 'index'
# for yaml_path in (data_path / 'hierarchy').glob('*.yaml'):
#     tree = build_tree_with_stats(yaml_path, index_path, duckdb_conn)
#     with (data_path / 'hierarchy' / f"{yaml_path.stem}.json").open('w') as f:
#         f.write(tree.to_json())

In [ ]:
# # run this only once to generate the TPC-DS data
import duckdb

con = duckdb.connect(database='./tpcds/tpcds.db')
# con = duckdb.connect(
#     database='./cube-project/data/tpcds.db',
#     read_only=True,
# )
# con.execute('INSTALL tpcds;')
# con.execute('LOAD tpcds;')
# con.execute("CALL dsdgen(sf = 1);")  # run only once generate data with scale factor 1 (1GB)

In [ ]:
df = con.execute("""SELECT ca_zip, Sum(cs_sales_price) 
FROM   catalog_sales, 
       customer, 
       customer_address, 
       date_dim 
WHERE  cs_bill_customer_sk = c_customer_sk 
       AND c_current_addr_sk = ca_address_sk 
       AND ( Substr(ca_zip, 1, 5) IN ( '85669', '86197', '88274', '83405', 
                                       '86475', '85392', '85460', '80348', 
                                       '81792' ) 
              OR ca_state IN ( 'CA', 'WA', 'GA' ) 
              OR cs_sales_price > 500 ) 
       AND cs_sold_date_sk = d_date_sk 
       AND d_qoy = 1 
       AND d_year = 1998 
GROUP  BY ca_zip 
ORDER  BY ca_zip
LIMIT 100; 
""").fetch_df()
df.head()

# Schema Graph

In [1]:
import sys
from pathlib import Path

proj_path = Path().resolve()
sys.path.append(str(proj_path / 'src'))

from src.graph_vis import display_graph
db_type = 'tutorial'  # 'tutorial' or 'tpcds'
data_path = proj_path / 'data' / db_type

In [4]:
# spring (default): Fruchterman-Reingold force-directed layout.
# kamada_kawai: Kamada-Kawai force-directed layout.
# circular: nodes positioned on a circle.
# shell: concentric shells (fact, dimensions, attributes, measures).
# spectral: based on the graph Laplacian eigenvectors.
# spiral: nodes arranged along an Archimedean spiral.
# random: random placement with a repeatable seed. Legacy Cytoscape names such as radial or cose are automatically mapped to the closest NetworkX algorithm.

display_graph(data_path, layout='kamada_kawai', height="1000px", width="1200px", 
              legend_toggles_labels=False, 
              node_opacity=1.0,
              node_spacing={'measure': 1.2, 'dimension': 1.1, 'attribute': 0.8, 'default': 0.9},
              node_properties={'fontsize': {'fact': 16, 'dimension': 14, 'default': 14}})

2025-11-27 01:56:50.004 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_date.json
2025-11-27 01:56:50.006 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_product.json
2025-11-27 01:56:50.007 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_store.json
2025-11-27 01:56:50.008 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/measure_sales.json


Output()

In [5]:
from src.schema_processor import SchemaExplorer

data_path = proj_path / 'data' / 'tutorial' # 'tutorial' or 'tpcds'
explorer = SchemaExplorer(data_path)
print(explorer.get_facts())
print(explorer.get_schema('star', 'fact_sales'))
print()
# TODO: need from/target searching
attr_results = explorer.search_attribute('star', 'fact_sales', 'year')  # d_fy_year, s_store_id
attr_results

2025-11-27 01:56:55.417 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_date.json
2025-11-27 01:56:55.420 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_product.json
2025-11-27 01:56:55.421 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_store.json
2025-11-27 01:56:55.422 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/measure_sales.json


{'fact_sales'}
[{'id': 'fact_sales', 'type': 'fact'}, {'id': 'dim_date', 'type': 'dimension'}, {'id': 'dim_product', 'type': 'dimension'}, {'id': 'dim_store', 'type': 'dimension'}]



[{'dimension': 'dim_date',
  'attribute': 'year',
  'path': [{'type': 'dimension', 'name': 'dim_date', 'label': 'dim_date'},
   {'type': 'level', 'name': 'date', 'label': 'Date'},
   {'type': 'level', 'name': 'month', 'label': 'Month'},
   {'type': 'level', 'name': 'quarter', 'label': 'Quarter'},
   {'type': 'attribute', 'name': 'year', 'label': 'Year'}],
  'stats': {'count': 365,
   'null_count': 0,
   'distinct_count': 1,
   'min': 2024,
   'max': 2024,
   'range': [2024, 2024],
   'dtype': 'integer',
   'unique_values': {'type': 'list', 'values': [2024]}}}]

In [ ]:
import random
from src.unique_index import UniqueIndex
path = './data/tpcds/index/date_dim__d_fy_year'
assert Path(path).exists(), f"Index path {path} does not exist."
idx = UniqueIndex(path, fast=False)

x = random.sample(list(iter(idx)), k=1)[0]
print("Search for:", x)
o = explorer.search_value('star', 'store_sales', 'd_fy_year', x)
print("Found:", o[0]['found'])
o

In [ ]:
o = explorer.search_measure('store_sales', 'sales_price')
o

----

## About the TPC-DS queries

In [ ]:
from pathlib import Path
from collections import defaultdict
from src.schema_processor import SchemaExplorer
queries_path = Path('./queries/tpcds')

fact2queries = defaultdict(list)
queries2fact = defaultdict(set)
for qpath in queries_path.glob('*.sql'):
    with qpath.open() as f:
        sql = f.read()
    for fact in SchemaExplorer.tpcds_facts:
        if fact in sql.lower():
            queries2fact[qpath.stem].add(fact)

for query, facts in queries2fact.items():
    # set the number of the facts as key, more or equal to 4 make them one group
    if len(facts) >= 4:
        fact2queries['4+'].append(query)
    else:
        fact2queries[str(len(facts))].append(query)

In [ ]:
for k, v in sorted(fact2queries.items(), key=lambda x: (int(x[0]) if x[0].isdigit() else 99)):
    print(f"{k}: {len(v)}")

In [ ]:
queries2fact['query15']

In [ ]:
facts_queries_by_numbers: dict[str, dict[str, list[str]]] = defaultdict(dict)
for fact in SchemaExplorer.tpcds_facts:
    for number, queries in fact2queries.items():
        if facts_queries_by_numbers[fact].get(number) is None:
            facts_queries_by_numbers[fact][number] = []
        facts_queries_by_numbers[fact][number].extend(queries)

In [ ]:
sorted(facts_queries_by_numbers['store_sales']['1'])[:5]

In [ ]:
# plot the distribution of number of fact tables per query
# make the number in the center of the bars
# if number is 4 or more, put it in 4+
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

fact_counts = [len(v) for v in queries2fact.values()]
fact_counts = [4 if x >= 4 else x for x in fact_counts]
fig, ax = plt.subplots()

sns.histplot(fact_counts, discrete=True, shrink=0.9, ax=ax)
ax.set_xlabel('Number of Fact Tables in a Query')
ax.set_ylabel('Number of Queries')
ax.set_title('Distribution of Number of Fact Tables per Query')
plt.xticks(ticks=[1, 2, 3, 4], labels=['1', '2', '3', '4+'])
plt.grid(axis='y')
plt.show()

---

In [1]:
import sys
from pathlib import Path

proj_path = Path().resolve()
sys.path.append(str(proj_path / 'src'))

db_type = 'tutorial'  # 'tutorial' or 'tpcds'
data_path = proj_path / 'data' / db_type

from src.schema_processor import SchemaExplorer
explorer = SchemaExplorer(data_path, schema_type='star')

2025-11-29 22:39:30.030 | INFO     | src.schema_processor:_load_hierarchies:169 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/tutorial/hierarchy/dim_date.json
2025-11-29 22:39:30.031 | INFO     | src.schema_processor:_load_hierarchies:169 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/tutorial/hierarchy/dim_product.json
2025-11-29 22:39:30.031 | INFO     | src.schema_processor:_load_hierarchies:169 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/tutorial/hierarchy/dim_store.json
2025-11-29 22:39:30.031 | INFO     | src.schema_processor:_load_hierarchies:169 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/tutorial/hierarchy/measure_sales.json


In [2]:
explorer.get_facts()

['fact_sales']

In [3]:
explorer.get_schema('fact_sales')

[{'name': 'fact_sales',
  'type': 'fact',
  'measures': ['total_units_sold',
   'total_sales_amount',
   'average_unit_price',
   'total_transactions',
   'average_order_value',
   'unique_products_sold',
   'active_stores',
   'active_days',
   'sales_per_store',
   'sales_per_day',
   'revenue_per_unit'],
  'fks': ['fact_sales.date_key = dim_date.date_key',
   'fact_sales.product_key = dim_product.product_key',
   'fact_sales.store_key = dim_store.store_key']},
 {'name': 'dim_date',
  'type': 'dimension',
  'attributes': ['date_key', 'date', 'week', 'month', 'quarter', 'year']},
 {'name': 'dim_product',
  'type': 'dimension',
  'attributes': ['product_key',
   'product_name',
   'brand',
   'type',
   'category',
   'department',
   'marketing_group']},
 {'name': 'dim_store',
  'type': 'dimension',
  'attributes': ['store_key',
   'store_name',
   'sales_manager',
   'sales_district',
   'city',
   'state']}]

In [4]:
explorer.search_attribute('fact_sales', 'state')

[{'dimension': 'dim_store',
  'attribute': 'state',
  'path': [{'type': 'dimension', 'name': 'dim_store', 'label': 'dim_store'},
   {'type': 'level', 'name': 'store_name', 'label': 'Store'},
   {'type': 'level', 'name': 'city', 'label': 'City'},
   {'type': 'attribute', 'name': 'state', 'label': 'State'}],
  'stats': {'count': 20,
   'null_count': 0,
   'distinct_count': 4,
   'min': None,
   'max': None,
   'range': None,
   'dtype': 'string',
   'unique_values': {'type': 'list',
    'values': ['California', 'Florida', 'New York', 'Texas']}}}]

In [5]:
explorer.search_value('fact_sales', 'week', 3)

True

# Agent

https://platform.openai.com/docs/guides/latest-model

https://openai.github.io/openai-agents-python/examples/

In [ ]:
import json
from agents import Agent, ModelSettings, function_tool
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
_ = load_dotenv(find_dotenv())

from src.schema_processor import SchemaExplorer
from pathlib import Path

class OutputList(BaseModel):
    results: list[str] = Field(..., description="List of results.")
    sql: str | None = Field(None, description="Generated SQL query, if applicable.")

data_path = Path("./data/tutorial")
explorer = SchemaExplorer(data_path, schema_type='star')

model_settings = ModelSettings(
    reasoning={"effort": "low"},  # low, medium, high
    verbosity='low',  # low, medium, high
    max_turns=30,
    response_format={
        "type": "json_object",
        "schema": OutputList.model_json_schema()
    },
)

@function_tool
def get_facts() -> set[str]:
    """returns the list of fact tables."""
    facts = explorer.get_facts()
    return facts

@function_tool
def get_schema_info(fact_table: str) -> list[dict]:
    """returns schema information for the specified fact table."""
    schema = explorer.get_schema(fact_table)
    return schema

@function_tool
def search_attribute(fact_table: str, attribute_name: str) -> list[dict]:
    """searches all hierarchy paths leading to the attribute for the fact schema.
    It returns a list of hierarchy paths and the statistics of the attribute.
    """
    results = explorer.search_attribute(fact_table, attribute_name)
    return results

@function_tool
def search_value_exists(fact_table: str, attribute_name: str, value: str) -> bool:
    """Return whether the attribute value exists in the specified fact table and attribute."""
    exists = explorer.search_value(fact_table, attribute_name, value)
    return exists

instructions_api = """You are an agent that helps users explore OLAP schema information. 
Derive the schema linking from natural language queries to structured fields:
1. measure: a column to be aggregated.
2. dimension: a column used to slice or group the measures.
3. filter: a condition to restrict the data. 

Format: Your output must strictly follow the JSON schema defined in OutputList.
1. Use `<measure:[str, table_name.column_name]>` to denote measures.
2. Use `<dimension:[str, table_name.column_name]>` to denote dimensions.
3. Use `<filter:[str, table_name.column_name]|val:[list[Any], value]>` to denote filters with specific values

For example:
Input: Show me the total sales by each brand for 2025.
Output:
{
    "results": [
        "measure:fact_table.total_sales_amount", 
        "dimension:custom.brand",
        "filter:date.year|val:[2025]"
    ]
}

You have access to the following tools:
1. get_facts(): returns the list of fact tables.
2. get_schema_info(fact_table): returns schema information for the specified fact table.
3. search_attribute(fact_table, attribute_name): searches all hierarchy paths leading to the attribute for the fact schema. It returns a list of hierarchy paths and the statistics of the attribute.
4. search_value_exists(fact_table, attribute_name, value): Return whether the attribute value exists.
Use these tools to answer user queries about the OLAP schema.
"""

instructions_sql = """You are an agent that helps users explore OLAP schema information. 
Derive the schema linking from natural language queries to structured fields first, then convert them to SQL:
1. measure: a column to be aggregated.
2. dimension: a column used to slice or group the measures.
3. filter: a condition to restrict the data. 

Format: Your output must strictly follow the JSON schema defined in OutputList.
1. Use `<measure:[str, table_name.column_name]>` to denote measures.
2. Use `<dimension:[str, table_name.column_name]>` to denote dimensions.
3. Use `<filter:[str, table_name.column_name]|val:[list[Any], value]>` to denote filters with specific values

For example:
Input: Show me the total sales by each brand for 2025.
Output:
{
    "results": [
        "measure:fact_table.total_sales_amount", 
        "dimension:custom.brand",
        "filter:date.year|val:[2025]"
    ]
    "sql": "SELECT custom.brand, SUM(fact_table.total_sales_amount) FROM fact_table JOIN custom ON fact_table.custom_id = custom.custom_key JOIN date ON fact_table.date_id = date.id WHERE date.year = 2025 GROUP BY custom.brand"
}

You have access to the following tools:
1. get_facts(): returns the list of fact tables.
2. get_schema_info(fact_table): returns schema information for the specified fact table.
3. search_attribute(fact_table, attribute_name): searches all hierarchy paths leading to the attribute for the fact schema. It returns a list of hierarchy paths and the statistics of the attribute.
4. search_value_exists(fact_table, attribute_name, value): Return whether the attribute value exists.
Use these tools to answer user queries about the OLAP schema.
"""


agent_api = Agent(
    name="OLAP Agent",
    instructions=instructions_api,
    model="gpt-5-nano",
    tools=[
        get_facts,
        get_schema_info,
        search_attribute,
        search_value_exists,
    ],
    model_settings=model_settings,
)


agent_sql = Agent(
    name="OLAP Agent",
    instructions=instructions_sql,
    model="gpt-5-nano",
    tools=[
        get_facts,
        get_schema_info,
        search_attribute,
        search_value_exists,
    ],
    model_settings=model_settings,
)

2025-11-29 22:56:31.515 | INFO     | src.schema_processor:_load_hierarchies:169 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/tutorial/hierarchy/dim_date.json
2025-11-29 22:56:31.516 | INFO     | src.schema_processor:_load_hierarchies:169 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/tutorial/hierarchy/dim_product.json
2025-11-29 22:56:31.516 | INFO     | src.schema_processor:_load_hierarchies:169 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/tutorial/hierarchy/dim_store.json
2025-11-29 22:56:31.517 | INFO     | src.schema_processor:_load_hierarchies:169 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/tutorial/hierarchy/measure_sales.json


In [7]:
from agents import Runner

query = "Show me the total sales for the east sales district by each quarter of 2024."
inputs=[
    {
        "role": "user", 
        "content": query
    }
]
result_api = await Runner.run(agent_api, inputs)
result_sql = await Runner.run(agent_sql, inputs)

In [13]:
result_sql.final_output.replace('\\\n', ' ')

'{\n  "results": [\n    "measure:fact_sales.total_sales_amount",\n    "dimension:dim_date.quarter",\n    "filter:dim_date.year|val:[2024]",\n    "filter:dim_store.sales_district|val:[East]"\n  ],\n  "sql": "SELECT dim_date.quarter, SUM(fact_sales.total_sales_amount) AS total_sales_amount  FROM fact_sales  JOIN dim_date ON fact_sales.date_key = dim_date.date_key  JOIN dim_store ON fact_sales.store_key = dim_store.store_key  WHERE dim_date.year = 2024 AND dim_store.sales_district = \'East\'  GROUP BY dim_date.quarter  ORDER BY dim_date.quarter"\n}'

In [15]:
x = result_api.final_output
print(json.loads(x)['results'])
print("-----"*10)

x = result_sql.final_output.replace('\\\n', '')
print(json.loads(x)['results'])
print(json.loads(x)['sql'])

['measure:fact_sales.total_sales_amount', 'dimension:dim_date.quarter', 'dimension:dim_store.sales_district', 'filter:dim_date.year|val:[2024]', 'filter:dim_store.sales_district|val:[East]']
--------------------------------------------------
['measure:fact_sales.total_sales_amount', 'dimension:dim_date.quarter', 'filter:dim_date.year|val:[2024]', 'filter:dim_store.sales_district|val:[East]']
SELECT dim_date.quarter, SUM(fact_sales.total_sales_amount) AS total_sales_amount FROM fact_sales JOIN dim_date ON fact_sales.date_key = dim_date.date_key JOIN dim_store ON fact_sales.store_key = dim_store.store_key WHERE dim_date.year = 2024 AND dim_store.sales_district = 'East' GROUP BY dim_date.quarter ORDER BY dim_date.quarter


In [16]:
for item in result_api.new_items:
    if item.type == 'reasoning_item':
        print(f"[{item.type}]: OpenAI hides the content")
    if item.type == 'tool_call_item':
        print(f"[{item.type}]: {item.raw_item.name}")
    if item.type == 'tool_call_output_item':
        print(f"[{item.type}]: {item.raw_item['output']}")

[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_facts
[tool_call_output_item]: ['fact_sales']
[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_schema_info
[tool_call_output_item]: [{'name': 'fact_sales', 'type': 'fact', 'measures': ['total_units_sold', 'total_sales_amount', 'average_unit_price', 'total_transactions', 'average_order_value', 'unique_products_sold', 'active_stores', 'active_days', 'sales_per_store', 'sales_per_day', 'revenue_per_unit'], 'fks': ['fact_sales.date_key = dim_date.date_key', 'fact_sales.product_key = dim_product.product_key', 'fact_sales.store_key = dim_store.store_key']}, {'name': 'dim_date', 'type': 'dimension', 'attributes': ['date_key', 'date', 'week', 'month', 'quarter', 'year']}, {'name': 'dim_product', 'type': 'dimension', 'attributes': ['product_key', 'product_name', 'brand', 'type', 'category', 'department', 'marketing_group']}, {'name': 'dim_store', 'type': 'dimension', 'attributes': ['store_key', 'store_name', 'sales

In [17]:
for item in result_sql.new_items:
    if item.type == 'reasoning_item':
        print(f"[{item.type}]: OpenAI hides the content")
    if item.type == 'tool_call_item':
        print(f"[{item.type}]: {item.raw_item.name}")
    if item.type == 'tool_call_output_item':
        print(f"[{item.type}]: {item.raw_item['output']}")

[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_facts
[tool_call_output_item]: ['fact_sales']
[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_schema_info
[tool_call_output_item]: [{'name': 'fact_sales', 'type': 'fact', 'measures': ['total_units_sold', 'total_sales_amount', 'average_unit_price', 'total_transactions', 'average_order_value', 'unique_products_sold', 'active_stores', 'active_days', 'sales_per_store', 'sales_per_day', 'revenue_per_unit'], 'fks': ['fact_sales.date_key = dim_date.date_key', 'fact_sales.product_key = dim_product.product_key', 'fact_sales.store_key = dim_store.store_key']}, {'name': 'dim_date', 'type': 'dimension', 'attributes': ['date_key', 'date', 'week', 'month', 'quarter', 'year']}, {'name': 'dim_product', 'type': 'dimension', 'attributes': ['product_key', 'product_name', 'brand', 'type', 'category', 'department', 'marketing_group']}, {'name': 'dim_store', 'type': 'dimension', 'attributes': ['store_key', 'store_name', 'sales

In [28]:
i = 0
item = result.new_items[i]
result.new_items[i].__dict__

{'agent': Agent(name='OLAP Agent', handoff_description=None, tools=[FunctionTool(name='get_facts', description='returns the list of fact tables.', params_json_schema={'properties': {}, 'title': 'get_facts_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f164b4a1620>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='get_schema_info', description='returns schema information for the specified fact table.', params_json_schema={'properties': {'fact_table': {'title': 'Fact Table', 'type': 'string'}}, 'required': ['fact_table'], 'title': 'get_schema_info_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7f170e0256c0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_outp

In [29]:
item.raw_item.__dict__

{'id': 'rs_0213f8fff7e89212006927bc640240819fa12e4600062c4e08',
 'summary': [],
 'type': 'reasoning',
 'content': None,
 'encrypted_content': None,
 'status': None}

In [31]:
result.new_items[3].__dict__

{'agent': Agent(name='OLAP Agent', handoff_description=None, tools=[FunctionTool(name='get_facts', description='returns the list of fact tables.', params_json_schema={'properties': {}, 'title': 'get_facts_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7ff51c4e5a80>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='get_schema_info', description='returns schema information for the specified fact table.', params_json_schema={'properties': {'fact_table': {'title': 'Fact Table', 'type': 'string'}}, 'required': ['fact_table'], 'title': 'get_schema_info_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7ff51c4e5d00>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_outp